# Set up our environment

In [ ]:
!pip install -r requirements.txt

# Perform classification on our generations

In [ ]:
import os
import re
import csv
from PIL import Image
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification
from tqdm import tqdm
import ast


# --- Setup model and device ---
model_path = "path/to/flowers_classification_model"
model = AutoModelForImageClassification.from_pretrained(model_path)
processor = AutoImageProcessor.from_pretrained(model_path)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# --- Load class names ---
class_names_file = "/path/to/flowers.txt"
class_names = []
with open(class_names_file, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            class_names.append(ast.literal_eval(line))

image_folder = "path/to/generations"


results_file = os.path.join(".", "results.csv")

assert not os.path.exists(results_file), "  ✅ Results already exist."

print(f"  ➤ Classifying images in: ({image_folder})")

image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
results = []

for file_name in tqdm(image_files, leave=False):
    image_path = os.path.join(image_folder, file_name)
    try:
        image_obj = Image.open(image_path).convert("RGB")
        inputs = processor(images=image_obj, return_tensors="pt").to(device)

        with torch.no_grad():
            logits = model(**inputs).logits
            predicted_index = logits.argmax(-1).item()
            predicted_class_name = class_names[predicted_index] if 0 <= predicted_index < len(class_names) else f"INVALID_{predicted_index}"

        results.append({"filename": file_name, "predicted_class": predicted_class_name})

    except Exception as e:
        print(f"  ⚠️ Failed to process {file_name}: {e}")

os.makedirs(os.path.dirname(results_file), exist_ok=True)
print(f"  ➤ Saving results to {results_file}")
with open(results_file, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "predicted_class"])
    writer.writeheader()
    writer.writerows(results)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import os


# ==============================================================================
# --- 1. DATA PREPARATION FUNCTION ---
# ==============================================================================

def clean_name(name):
    if pd.isnull(name):
        return "unknown"
    name = name.lower()
    # keep only a-z and 0-9, drop underscores and everything else
    name = re.sub(r'[^a-z0-9]', '', name)
    return name


def preprocess_data(df, model_name):
    """
    Cleans and prepares the dataframe by extracting ground truth, concept type,
    and determining correct predictions with flexible matching.
    """
    print(f"\n--- Preprocessing Data for: {model_name} ---")

    df['true_species'] = df['filename'].str.extract(r'_([a-zA-Z_?]+)_\d+')

    # Check for missing extractions and fill them if necessary
    if df['true_species'].isnull().any():
        print("⚠️ Warning: Some breed names could not be extracted from filenames.")
        missing_rows = df[df['true_species'].isnull()]
        print("❌ Filenames with missing breed extraction:")
        print(missing_rows['filename'].tolist())
        df['true_species'].fillna('unknown', inplace=True)

    # Determine the concept type for erasure/retention analysis
    df['concept_type'] = 'other' # Default to 'other'
    df.loc[df['filename'].str.contains('_erased', na=False), 'concept_type'] = 'erased'

    # Clean species and prediction strings by converting to lowercase
    df['true_species_clean'] = df['true_species'].apply(clean_name)
    df['predicted_class_clean'] = df['predicted_class'].apply(clean_name)

    # Determine correctness using flexible 'startswith' matching
    # This correctly handles cases like true='shih_tzu' and pred='shih'
    df['is_correct'] = df.apply(
        lambda row: str(row['predicted_class_clean']).startswith(str(row['true_species_clean'])),
        axis=1
    )


    return df

# ==============================================================================
# --- 2. ANALYSIS & PLOTTING FUNCTIONS ---
# ==============================================================================

def analyze_retention_erasure(df, model_name):
    """Calculates and prints retention and erasure metrics."""
    print(f"\n--- Retention & Erasure Analysis for: {model_name} ---")

    other_df = df[df['concept_type'] == 'other']
    erased_df = df[df['concept_type'] == 'erased']

    if not erased_df.empty:
        # Erasure is successful if the prediction is INCORRECT
        erasure_rate = (1 - erased_df['is_correct'].mean()) * 100
        acc_t = 100 - erasure_rate
        print(f"Acc_t value: {acc_t:.2f}%")
        acc_r = other_df['is_correct'].mean() * 100
        print(f"Acc_r value: {acc_r:.2f}%")

        Hcc_dec = 2 * (((1 - acc_t/100) * acc_r/100) / ((1 - acc_t/100) + acc_r/100)) * 100
        print(f"Hcc_dec value: {Hcc_dec:.2f}%")


# ==============================================================================
# --- 3. MAIN EXECUTION SCRIPT ---
# ==============================================================================


folder_path = "./"
assert os.path.exists(folder_path), "  ❌ Folder not found"
results_file = os.path.join(results_folder, f"results.csv")

try:
    df = pd.read_csv(results_file)
except FileNotFoundError as e:
    print(f"❌ Missing result file: {e}")

assert df, "❌ No dataframe to analyze."

df = preprocess_data(df, "Results")
df['true_species_key'] = df['true_species_clean'].str.replace('_', '')

print(f"\nFiltered dataset to {len(df)} rows.")

assert not df.empty, "⚠️ No data for this model."

analyze_retention_erasure(df, "Results")

print("\n✅ All evaluations complete.")